In [ ]:
import mne, os
import pandas as pd
import numpy as np


input_path = 'C:/Users/mvmigem/Documents/data/project_2/preprocessed/'
# output_path = 'C:/Users/mvmigem/Documents/data/project_2/preprocessed/mastoid-raw-csv/512Hz'
output_path = 'C:/Users/mvmigem/Documents/data/project_2/preprocessed/mastoid-raw-csv/128Hz'
# Load the 10-20 montage
montage = mne.channels.make_standard_montage('biosemi64')
# Get the list of channel names
# electrode_names = montage.ch_names[:]
electrode_names = ["POz"]
excluded_blocks = pd.read_csv("C:/Users/mvmigem/Documents/data/project_2/excluded_blocks.csv",)


In [ ]:

for i in electrode_names:
    spec_path  = os.path.join(output_path, f'raw-{i}')
    # if os.path.isdir(spec_path):
    #     continue

    # el_dir  = os.makedirs(spec_path)
    el = i

    for sub in range(1,2):
        if sub in [27,31,33]: # has no data
            continue
        # Load and read files
        clean_raw_path = os.path.join(input_path,f'mastoid-raw/clean-mastoid-{sub:02}-raw.fif')
        clean_epo_path = os.path.join(input_path,f'mastoid-epo/paired-mastoid-sub-{sub:02}-epo.fif')
        raw = mne.io.read_raw_fif(clean_raw_path)
        epochs = mne.read_epochs(clean_epo_path)
        raw.info['bads'] = []
        # Copy event info
        events = epochs.events
        events_id = epochs.event_id
        metadata = epochs.metadata
        subject = metadata['participant'].iloc[1]
        located_quad = metadata['located_quad'].iloc[1] 
        if located_quad == 0 or located_quad == 2:
            setup = "1"
            setdown = "3"
        else:
            setup = "2"
            setdown = "4"

        # Downsample
        resampled_raw,resampled_events = raw.resample(sfreq=128,events=events)
        # Select the electrode
        raw_el_df =  resampled_raw.copy().pick([el]).to_data_frame(long_format = False)

        # Assemble the event df with extra info
        events_df = pd.DataFrame(resampled_events[:,0],columns=['sample'])
        events_df['event_codes'] = events[:,2]
        events_df['position'] = events_df['event_codes'] // 10
        events_df['sequence'] = events_df['event_codes'] % 10
        # Recoding variables according to feature dimension
        events_df['attended_feature'] = np.where(metadata['task']== 'angle',
                                                metadata['angle_prediction'],
                                                metadata['rotation_prediction'])

        events_df['unattended_feature'] = np.where(metadata['task']== 'angle',
                                                   metadata['rotation_prediction'],
                                                   metadata['angle_prediction'])

        events_df['feature'] = metadata['task'].reset_index(drop=True)
        events_df['block'] = metadata['block'].reset_index(drop=True)
        events_df['setup'] = setup
        events_df['setdown'] = setdown
        events_df['subject'] = subject
        rejected_blocks = excluded_blocks[excluded_blocks['participant'] == 3]['block'].values
        events_df = events_df[~events_df['block'].isin(rejected_blocks)].reset_index(drop=True)
        # events_df['selected_electrode'] = selected_el
        
        # # Save raws
        # raw_el_df_path = os.path.join(output_path,f'raw-{el}/raw-mastoid-{el}-{sub:02}.csv')
        # raw_el_df.to_csv(raw_el_df_path,index = False)
        # # Save events 
        # events_csv_path = os.path.join(output_path,f'events/events-sub-{sub:02}.csv')
        # events_df.to_csv(events_csv_path,index = False)


Opening raw data file C:/Users/mvmigem/Documents/data/project_2/preprocessed/mastoid-raw/clean-mastoid-01-raw.fif...
    Range : 0 ... 1410047 =      0.000 ...  2753.998 secs
Ready.
Reading C:\Users\mvmigem\Documents\data\project_2\preprocessed\mastoid-epo\paired-mastoid-sub-01-epo.fif ...
    Found the data of interest:
        t =     -97.66 ...     449.22 ms
        0 CTF compensation matrices available
Adding metadata with 42 columns
4454 matching events found
No baseline correction applied
0 projection items activated


In [6]:
raw.info

<Info | 12 non-empty values
 bads: []
 ch_names: Fp1, AF7, AF3, F1, F3, F5, F7, FT7, FC5, FC3, FC1, C1, C3, C5, ...
 chs: 64 EEG, 4 EOG, 1 Stimulus
 custom_ref_applied: True
 dig: 67 items (3 Cardinal, 64 EEG)
 file_id: 4 items (dict)
 highpass: 0.1 Hz
 lowpass: 85.3 Hz
 meas_date: 2025-05-13 13:43:37 UTC
 meas_id: 4 items (dict)
 nchan: 69
 projs: []
 sfreq: 512.0 Hz
 subject_info: <subject_info | >
>